# 🗺️ Uri Geodata Exploration - REAL Swiss Land Cover Data

**Dataset:** REAL Swiss land cover data from Kanton Uri WMS  
**Records:** 58,689 land parcels (bodenbedeckung layer)  
**Source:** https://geo.ur.ch/wms (official Swiss government data)  
**Updated:** March 28, 2026

## Learning Goals
- Load and explore real Swiss geospatial data
- Understand land cover type distributions
- Analyze parcel sizes and quality
- Prepare features for ML classification

## Dataset Overview
- **Total Features:** 112,145 (3 layers)
  - Bodenbedeckung (land cover): 58,689 parcels
  - Einzelobjekte (individual objects): 53,445 features
  - Biotope (protected areas): 11 features
- **Columns:** qualitaet, art, bfsnr, art_nr, shape_length, shape_area, flaeche_m2
- **ML Task:** Multi-class classification (predict land cover type from features)

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

# Load the Uri geodata (bodenbedeckung layer - main dataset)
df = pd.read_csv('../data/processed/uri_geodata_ml_ready.csv')

print(f'Dataset loaded: {len(df):,} records')
print(f'Columns: {len(df.columns)}')
print(f'\nColumns: {df.columns.tolist()}')

## 1. Basic Data Exploration

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Dataset info
df.info()

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df[missing_df['Missing Count'] > 0]

## 2. Land Cover Type Distribution (Target Variable)

In [ ]:
# Count by land cover type (art column)
print('🌍 Land Cover Type Distribution:')
print('='*60)
land_cover_counts = df['art'].value_counts()

for cover_type, count in land_cover_counts.items():
    pct = count / len(df) * 100
    print(f'{cover_type:40s}: {count:6,} ({pct:5.1f}%)')

In [ ]:
# Bar chart: Land cover types
plt.figure(figsize=(14, 8))
top_types = land_cover_counts.head(15)
bars = plt.bar(range(len(top_types)), top_types.values, alpha=0.7, color='steelblue')
plt.xlabel('Land Cover Type', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Land Cover Type Distribution in Uri (Top 15)', fontsize=14, fontweight='bold')
plt.xticks(range(len(top_types)), top_types.index, rotation=45, ha='right', fontsize=10)

# Add value labels on bars
for i, v in enumerate(top_types.values):
    plt.text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)

plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Pie chart: Top 5 land cover types
top5 = land_cover_counts.head(5)
other = land_cover_counts.iloc[5:].sum()

plt.figure(figsize=(10, 10))
sizes = list(top5.values) + [other]
labels = list(top5.index) + ['Other']
colors = plt.cm.Set3(np.linspace(0, 1, len(sizes)))

plt.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', 
        startangle=90, pctdistance=0.85)
plt.title('Land Cover Type Distribution (Top 5 + Other)', fontsize=14, fontweight='bold')
plt.axis('equal')
plt.tight_layout()
plt.show()

## 3. Parcel Size Analysis

In [ ]:
# Parcel size statistics
print('📏 Parcel Size Statistics (flaeche_m2):')
print('='*60)
print(f'Total parcels: {len(df):,}')
print(f'Average size: {df["flaeche_m2"].mean():,.0f} m² ({df["flaeche_m2"].mean()/10000:.2f} hectares)')
print(f'Median size: {df["flaeche_m2"].median():,.0f} m²')
print(f'Largest parcel: {df["flaeche_m2"].max():,.0f} m² ({df["flaeche_m2"].max()/10000:.2f} hectares)')
print(f'Smallest parcel: {df["flaeche_m2"].min():,.0f} m²')
print(f'Std deviation: {df["flaeche_m2"].std():,.0f} m²')

In [ ]:
# Histogram: Parcel size distribution (log scale for better visualization)
plt.figure(figsize=(12, 6))
# Clip extreme values for better visualization
sizes_clipped = df['flaeche_m2'].clip(0, 50000)
plt.hist(sizes_clipped, bins=100, alpha=0.7, edgecolor='black', color='forestgreen')
plt.xlabel('Parcel Size (m²) - clipped to 50,000', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Distribution of Parcel Sizes', fontsize=14, fontweight='bold')
plt.axvline(df['flaeche_m2'].median(), color='red', linestyle='--', linewidth=2,
            label=f'Median: {df["flaeche_m2"].median():,.0f} m²')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Box plot: Parcel size by land cover type (top 10)
top10_types = land_cover_counts.head(10).index
df_top10 = df[df['art'].isin(top10_types)]

plt.figure(figsize=(14, 8))
df_top10.boxplot(column='flaeche_m2', by='art', rot=45, ax=plt.gca())
plt.suptitle('Parcel Size Distribution by Land Cover Type (Top 10)', fontsize=14, fontweight='bold')
plt.xlabel('Land Cover Type', fontsize=12)
plt.ylabel('Parcel Size (m²)', fontsize=12)
plt.yscale('log')  # Log scale for better visualization
plt.tight_layout()
plt.show()

## 4. Quality Distribution

In [ ]:
# Quality statistics
print('📊 Quality Distribution (qualitaet):')
print('='*60)
quality_counts = df['qualitaet'].value_counts()

for quality, count in quality_counts.items():
    pct = count / len(df) * 100
    print(f'{quality:10s}: {count:6,} ({pct:5.1f}%)')

In [ ]:
# Bar chart: Quality distribution
plt.figure(figsize=(10, 6))
bars = plt.bar(range(len(quality_counts)), quality_counts.values, alpha=0.7, color='coral')
plt.xlabel('Quality Code', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Data Quality Distribution', fontsize=14, fontweight='bold')
plt.xticks(range(len(quality_counts)), quality_counts.index)

# Add value labels
for i, v in enumerate(quality_counts.values):
    plt.text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)

plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 5. Numeric Feature Analysis

In [ ]:
# Statistical summary of numeric columns
numeric_cols = ['shape_length', 'shape_area', 'flaeche_m2', 'art_nr']
df[numeric_cols].describe()

In [ ]:
# Correlation matrix
corr_cols = ['shape_length', 'shape_area', 'flaeche_m2', 'art_nr']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', square=True, linewidths=0.5, 
            cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Land Cover Type vs Parcel Size

In [ ]:
# Average parcel size by land cover type
size_by_type = df.groupby('art')['flaeche_m2'].agg(['mean', 'median', 'std', 'count']).sort_values('mean', ascending=False)
size_by_type.columns = ['Mean Size (m²)', 'Median Size (m²)', 'Std Dev', 'Count']
size_by_type['Mean Size (m²)'] = size_by_type['Mean Size (m²)'].round(0)
size_by_type['Median Size (m²)'] = size_by_type['Median Size (m²)'].round(0)
size_by_type['Std Dev'] = size_by_type['Std Dev'].round(0)

print('📊 Parcel Size by Land Cover Type:')
print('='*80)
print(size_by_type.head(15))

In [ ]:
# Bar chart: Average parcel size by land cover type
plt.figure(figsize=(14, 8))
avg_sizes = df.groupby('art')['flaeche_m2'].mean().sort_values(ascending=False).head(15)
bars = plt.bar(range(len(avg_sizes)), avg_sizes.values, alpha=0.7, color='darkorange')
plt.xlabel('Land Cover Type', fontsize=12)
plt.ylabel('Average Parcel Size (m²)', fontsize=12)
plt.title('Average Parcel Size by Land Cover Type (Top 15)', fontsize=14, fontweight='bold')
plt.xticks(range(len(avg_sizes)), avg_sizes.index, rotation=45, ha='right', fontsize=10)

# Add value labels
for i, v in enumerate(avg_sizes.values):
    plt.text(i, v, f'{v:,.0f}', ha='center', va='bottom', fontsize=8)

plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 7. Feature Engineering Opportunities

In [ ]:
# Create size categories
def categorize_size(size):
    if size < 100:
        return 'tiny (<100m²)'
    elif size < 500:
        return 'small (100-500m²)'
    elif size < 2000:
        return 'medium (500-2000m²)'
    elif size < 10000:
        return 'large (2000-10000m²)'
    else:
        return 'very_large (>10000m²)'

df['size_category'] = df['flaeche_m2'].apply(categorize_size)

print('📏 Size Category Distribution:')
print('='*60)
size_cat_counts = df['size_category'].value_counts()
for cat, count in size_cat_counts.items():
    pct = count / len(df) * 100
    print(f'{cat:25s}: {count:6,} ({pct:5.1f}%)')

In [ ]:
# Cross-tabulation: Size category vs land cover type
print('\n🔍 Size Category by Land Cover Type (Top 5):')
print('='*80)
top5_types = land_cover_counts.head(5).index
df_top5 = df[df['art'].isin(top5_types)]
cross_tab = pd.crosstab(df_top5['art'], df_top5['size_category'], margins=True, margins_name='Total')
print(cross_tab)

## 8. ML Task Definition

In [ ]:
print('='*80)
print('🤖 ML TASK DEFINITION')
print('='*80)
print()
print('📊 DATASET SUMMARY:')
print(f'  - Total records: {len(df):,}')
print(f'  - Total features: {len(df.columns)}')
print(f'  - Land cover types: {df["art"].nunique()}')
print()
print('🎯 RECOMMENDED ML TASK:')
print('  Multi-class Classification')
print('  - Target: art (land cover type)')
print('  - Features: flaeche_m2, qualitaet, shape_length, shape_area, art_nr')
print('  - Algorithm: Decision Tree (build from scratch!)')
print()
print('📈 ALTERNATIVE ML TASKS:')
print('  1. Regression: Predict parcel size (flaeche_m2)')
print('     - Features: land cover type, quality, shape metrics')
print('     - Algorithm: Linear Regression (build from scratch!)')
print()
print('  2. Binary Classification: Large vs Small parcels')
print('     - Target: size_category (large/very_large vs others)')
print('     - Features: land cover type, quality, location')
print('     - Algorithm: Logistic Regression (build from scratch!)')
print()
print('⚠️  DATA CHARACTERISTICS:')
print(f'  - Class imbalance: Top 5 types = {(land_cover_counts.head(5).sum()/len(df)*100):.1f}% of data')
print('  - Missing values: Check qualitaet column')
print('  - Feature scaling needed: shape_length, shape_area, flaeche_m2')
print()
print('📚 NEXT STEPS:')
print('  1. Preprocess data (encode categorical variables)')
print('  2. Split data (train/test split: 80/20)')
print('  3. Implement Decision Tree from scratch')
print('  4. Train and evaluate model')
print('  5. Analyze feature importance')
print()
print('='*80)

## 9. Data Preparation for ML

In [ ]:
# Preview: Prepare features for ML
print('🔧 PREPARING FEATURES FOR ML:')
print('='*60)

# Select features
features = ['flaeche_m2', 'qualitaet', 'shape_length', 'shape_area']
target = 'art'

print(f'Features: {features}')
print(f'Target: {target}')
print()

# Check data types
print('Data Types:')
for col in features + [target]:
    print(f'  {col:20s}: {df[col].dtype}')
print()

# Encoding needed for categorical variables
categorical = ['qualitaet', 'art']
numeric = ['flaeche_m2', 'shape_length', 'shape_area']

print('Categorical columns (need encoding):')
for col in categorical:
    print(f'  {col}: {df[col].nunique()} unique values')
print()

print('Numeric columns (need scaling):')
for col in numeric:
    print(f'  {col}: range [{df[col].min():.2f}, {df[col].max():.2f}]')
print()

print('✅ Ready for preprocessing!')

## 10. Summary & Next Steps

### ✅ Completed:
- Loaded Uri geodata (58,689 land parcels)
- Analyzed land cover type distribution (27 types)
- Examined parcel size statistics (avg: 18,343 m²)
- Reviewed quality distribution (PN: 58.7%, AV93: 33.7%)
- Identified ML task: Multi-class classification
- Created size categories for feature engineering

### 📊 Key Insights:
1. **Dominant land covers:** Gebaeude (buildings), bestockter Wald (forest), Strasse (roads)
2. **Size variation:** Huge range from 9 m² to 12.9 million m²
3. **Quality:** Most data is high quality (PN, AV93)
4. **Correlations:** shape_area and flaeche_m2 highly correlated (as expected)

### 🚀 Next Steps:
1. **Preprocessing:**
   - Encode categorical variables (qualitaet, art)
   - Scale numeric features (StandardScaler)
   - Handle missing values if any

2. **Model Implementation:**
   - Implement Decision Tree from scratch in `src/models/decision_tree.py`
   - Use entropy or gini impurity for splits
   - Build recursively with max_depth parameter

3. **Training & Evaluation:**
   - Split data 80/20 (train/test)
   - Train decision tree
   - Evaluate with accuracy, precision, recall, F1
   - Analyze feature importance

### 📚 Resources:
- See `docs/learning_resources.md` for ML courses
- See `docs/build_from_scratch.md` for implementation guidelines
- Implement Decision Tree in `src/models/decision_tree.py`